In [1]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2

In [2]:
import optuna
import pickle
from functools import partial
from pathlib import Path

from simulator.simulation.modules import Campaign
from simulator.simulation.utils_visualization import data_prep_vis, plot_history_article
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import autobidder_check

In [3]:
import pandas as pd

In [4]:
from simulator.model.rlb_dp_bidder import RLBDPBidder

In [5]:
auction_mode = "FPA"  # or "VCG"
best_params_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"
best_models_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"

# metric to optimize: CPC_REL / RMSE / SCR
metric = "SCR"
n_trials = 10

In [6]:
data_config = {
    "train": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_train_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_train_final.csv",
    },
    "test": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_test_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_test_final.csv",
    },
}

data_config

{'train': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_train_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_train_final.csv'},
 'test': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_test_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_test_final.csv'}}

In [7]:
stats_path = data_config['train']['stats_path']
campaigns_path = data_config['train']['campaigns_path']

In [8]:
stats_df = pd.read_csv(stats_path)

In [9]:
def objective_rlb_dp(trial, metric='RMSE_T', auction_mode='FPA'):

    max_bid = trial.suggest_float('max_bid', 10, 500, log=True)
    gamma = trial.suggest_float('gamma', 0.80, 1.00) 
    N_bound = trial.suggest_int('N_bound', 6, 72)
    B_bound = trial.suggest_int('B_bound', 1e3, 2e4, log=True)

    custom_params = {
        "max_bid": max_bid,
        "gamma": gamma,
        "model_path": None,
        "N_bound": N_bound,
        "B_bound": B_bound,
    }


    bidder = RLBDPBidder(custom_params)

    bidder.fit(stats_df)
    import os, uuid

    os.makedirs("tmp_models", exist_ok=True)
    TMP_MODEL_PATH = f"tmp_models/rlb_dp_trial{trial.number}_{uuid.uuid4().hex}.pkl"
    bidder.save_model(TMP_MODEL_PATH)

    # прогоняем через тот же пайплайн проверки
    res = autobidder_check(
        bidder=RLBDPBidder,
        params={
            "input_campaigns": campaigns_path,
            "input_stats": stats_path,
            "max_bid": max_bid,
            "gamma": gamma,
            "model_path": TMP_MODEL_PATH,
            "N_bound": N_bound,
            "B_bound": B_bound,
        },
        auction_mode=auction_mode,
    )

    print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")
    if metric == 'RMSE_T':
        return res['score'][1]
    elif metric == 'CPC_REL':
        return res['score'][0]
    elif metric == 'SCR':
        return res['score'][2]


def opt_search_rlb_dp(n_trials, metric='RMSE_T', auction_mode='FPA'):
    study = optuna.create_study(
        direction='maximize' if metric == 'SCR' else 'minimize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    
    study.optimize(
        partial(objective_rlb_dp, metric=metric, auction_mode=auction_mode),
        n_trials=n_trials,
        n_jobs=6
    )

    print('Best trial:')
    trial = study.best_trial
    print(f'  Value: {trial.value}')
    print('  Params: ')

    dict_path = f'best_params/rlb_dp_{metric.lower()}_{auction_mode}.pkl'
    params_dict = {}
    for key, value in trial.params.items():
        print(f'    {key}: {value}')
        params_dict[key] = value

    with open(dict_path, 'wb') as f:
        pickle.dump(params_dict, f)

    return study


def train_best_rlb_dp(best_params_path, model_path='rlb_dp_model_tuned.pkl'):
    """Обучить и сохранить модель с лучшими параметрами (после optuna)."""
    with open(best_params_path, 'rb') as f:
        best_params = pickle.load(f)

    custom_params = {
        "max_bid": best_params["max_bid"],
        "gamma": best_params["gamma"],
        "model_path": None,
        "N_bound": best_params["N_bound"],
        "B_bound": best_params["B_bound"],
    }

    bidder = RLBDPBidder(custom_params)
    bidder.fit(stats_df)
    bidder.save_model(model_path)
    return bidder


In [ ]:
study_rlb = opt_search_rlb_dp(n_trials, metric, auction_mode)

[I 2026-02-24 22:14:26,993] A new study created in memory with name: no-name-f5bada4e-869f-4a9e-adaa-0a069f2f73c1
Hours:   0%|          | 0/71 [00:00<?, ?it/s]









Hours:  13%|█▎        | 9/71 [00:00<00:00, 88.03it/s]









Hours:  25%|██▌       | 18/71 [00:01<00:02, 22.05it/s]





Hours: 100%|██████████| 22/22 [00:01<00:00, 20.14it/s] 





Hours:  37%|███▋      | 26/71 [00:01<00:03, 14.72it/s]





Hours:  41%|████      | 29/71 [00:01<00:03, 13.43it/s]





Hours:  44%|████▎     | 31/71 [00:02<00:03, 12.48it/s]





Hours:  46%|████▋     | 33/71 [00:02<00:03, 12.18it/s]





Hours:  49%|████▉     | 35/71 [00:02<00:03, 11.93it/s]





Hours:  52%|█████▏    | 37/71 [00:02<00:02, 11.72it/s]





Hours:  55%|█████▍    | 39/71 [00:02<00:02, 11.62it/s]





Hours:  58%|█████▊    | 41/71 [00:03<00:02, 10.84it/s]





Hours:  61%|██████    | 43/71 [00:03<00:02, 10.96it/s]





Hours:  63%|██████▎   | 45/71 [00:03<00:02, 11.00it/s]





Hours:  66%|██████▌   | 47/71 [00:03<00:02, 10.

CPC_REL: 428.4412721120784, rmse: 1.1951206197318784, SCR: 22077.253899984506


[I 2026-02-24 22:23:46,666] Trial 4 finished with value: 31727.700845685482 and parameters: {'max_bid': 101.83008737722197, 'gamma': 0.8418945293362943, 'N_bound': 51, 'B_bound': 5640}. Best is trial 4 with value: 31727.700845685482.


CPC_REL: 265.4659545328967, rmse: 1.1617862480480792, SCR: 31727.700845685482


Hours:  83%|████████▎ | 57/69 [00:01<00:00, 38.72it/s]

...

[I 2026-02-24 22:29:53,801] Trial 9 finished with value: 30845.145475150795 and parameters: {'max_bid': 151.6959047468111, 'gamma': 0.8284682594920014, 'N_bound': 11, 'B_bound': 2518}. Best is trial 0 with value: 32685.45488782264.


CPC_REL: 265.6450036378069, rmse: 1.194730493859638, SCR: 30845.145475150795


[I 2026-02-24 22:29:56,816] Trial 8 finished with value: 30951.14867324265 and parameters: {'max_bid': 102.22625654676501, 'gamma': 0.9548759185730638, 'N_bound': 17, 'B_bound': 3984}. Best is trial 0 with value: 32685.45488782264.


CPC_REL: 265.4526713564091, rmse: 1.1503262291341672, SCR: 30951.14867324265
Best trial:
  Value: 32685.45488782264
  Params: 
    max_bid: 275.0311519907557
    gamma: 0.9233823733827753
    N_bound: 22
    B_bound: 2076


FileNotFoundError: [Errno 2] No such file or directory: 'best_params/fpa_rlb_n10_rndm_42/scr.pkl'

In [13]:
# best_params_path = f'best_params/{best_params_subfolder}/{metric.lower()}.pkl'
best_params_path='best_params/rlb_dp_scr_FPA.pkl'
best_model_path = f'best_models/{best_models_subfolder}/{metric.lower()}.pkl'

rlb_bidder_best = train_best_rlb_dp(best_params_path, best_model_path)

Hours: 100%|██████████| 22/22 [00:00<00:00, 179.78it/s]


In [14]:
best_params_path

'best_params/rlb_dp_scr_FPA.pkl'

In [15]:
best_params_rlb = pd.read_pickle(best_params_path)
best_model_rlb_path = best_model_path

In [16]:
campaigns_path_test = data_config['test']['campaigns_path']
stats_path_test = data_config['test']['stats_path']

In [17]:
res = autobidder_check(
    bidder=RLBDPBidder,
    params = {
        "input_campaigns": campaigns_path_test,
        "input_stats": stats_path_test,
        "model_path": best_model_path,
        **best_params_rlb
    },
    auction_mode=auction_mode,
)

In [18]:
print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")

CPC_REL: 328.38205798055765, rmse: 1.2777773701938937, SCR: 32566.61670594752
